In [20]:
%%writefile square_integers.cu

#include <stdio.h>
#include "cuda_runtime.h"
#include "device_launch_parameters.h"
#define N 5


__global__ void square(int *a, int *result)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N)
    {
        int val = a[idx];
        result[idx] = val * val;
    }
}

int main()
{
    int host_a[N] = {1, 2, 3, 4, 5};
    int host_result[N];

    // device pointers
    int *device_a, *device_result;

    // allocate memory on device
    cudaMalloc((void **)&device_a, N * sizeof(int));
    cudaMalloc((void **)&device_result, N * sizeof(int));

    // copy input array from host to device
    cudaMemcpy(device_a, host_a, N * sizeof(int), cudaMemcpyHostToDevice);

    // launch kernel: one block, N threads (one per element)
    square<<<1, N>>>(device_a, device_result);
    cudaDeviceSynchronize();

    // copy result back from device to host
    cudaMemcpy(host_result, device_result, N * sizeof(int), cudaMemcpyDeviceToHost);

    // display the result
    printf("Input:  ");
    for (int i = 0; i < N; i++)
        printf("%d ", host_a[i]);
    printf("\n");

    printf("Squared: ");
    for (int i = 0; i < N; i++)
        printf("%d ", host_result[i]);
    printf("\n");

    // free memory
    cudaFree(device_a);
    cudaFree(device_result);

    return 0;
}

Overwriting square_integers.cu


In [21]:
!nvcc square_integers.cu -o square_integers

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [22]:
!./square_integers

Input:  1 2 3 4 5 
Squared: 1 4 9 16 25 
